# CNN Architectures
Today we will cover:
- Theory: LeNet, AlexNet, VGG
- Exercise: Train VGG on CIFAR-10
- Mini Project: Image Classifier

## Why Study CNN Architectures?
CNNs evolved over time to handle increasingly complex image tasks.
We will study three landmark architectures:

### LeNet (1998)
- Developed by Yann LeCun for digit recognition.
- Architecture: Convolution → Pooling → Fully Connected.
- Used for MNIST digit classification.
- Key Contribution: Showed CNNs can outperform traditional ML for images.

### AlexNet (2012)
- Won ImageNet competition.
- Architecture: 5 convolutional layers + 3 fully connected layers.
- Introduced ReLU activation, dropout, and GPU training.
- Key Contribution: Sparked the deep learning revolution.

### VGG (2014)
- Very deep network (16–19 layers).
- Used small 3x3 filters stacked together.
- Key Contribution: Showed depth improves accuracy, standardized design.

## Setup & Data Loading

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# CIFAR-10 dataset: 60,000 images, 10 classes
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64,
                                          shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64,
                                         shuffle=False)

classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')

## Exercise: Train VGG on CIFAR-10

In [ ]:
import torchvision.models as models

# Load VGG16 and adapt for CIFAR-10
vgg16 = models.vgg16(pretrained=False)
vgg16.classifier[6] = nn.Linear(4096, 10)  # adjust output layer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vgg16.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg16.parameters(), lr=0.001)

# Training loop (simplified)
for epoch in range(1, 3):  # train for 2 epochs for demo
    running_loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = vgg16(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch}, Loss: {running_loss/len(trainloader):.4f}")

## Mini Project: Image Classifier

In [ ]:
# Evaluate on test set
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = vgg16(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy on CIFAR-10 test set: {100 * correct / total:.2f}%")

# Try predicting on a single batch of images
import matplotlib.pyplot as plt
import numpy as np

def imshow(img):
    img = img / 2 + 0.5  # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

dataiter = iter(testloader)
images, labels = next(dataiter)
imshow(torchvision.utils.make_grid(images[:4]))
outputs = vgg16(images.to(device))
_, predicted = torch.max(outputs, 1)
print('Predicted:', ' '.join(f'{classes[predicted[j]]}' for j in range(4)))

## Visualizing VGG Architecture with Graphviz

In [ ]:
# Visualizing VGG Architecture with Graphviz
from torchviz import make_dot
import torchvision.models as models
import torch
from IPython.display import Image

# Load VGG16 model
vgg16 = models.vgg16(pretrained=False)

# Create a dummy input to trace the model
x = torch.randn(1, 3, 224, 224)
y = vgg16(x)

# Generate visualization
dot = make_dot(y, params=dict(vgg16.named_parameters()))
dot.format = 'png'
dot.render('vgg16_architecture')

# Display the diagram
Image(filename='vgg16_architecture.png')